### Guiding in prompts


In [14]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_community.chat_models import ChatOllama
import warnings
import json
warnings.filterwarnings("ignore")

llm = ChatOllama(
    model="mistral:latest",
    temperature=0.7
)

In [15]:
from langchain_core.prompts import PromptTemplate
#result= llm.invoke("Tell me a joke.")
result = llm.invoke("Tell me a joke. Generate the output in key-value pair format with the following keys: setup, punchline")
result

AIMessage(content=' {\n "setup": "Why don\'t scientists trust atoms?",\n "punchline": "Because they make up everything!"\n}', additional_kwargs={}, response_metadata={'model': 'mistral:latest', 'created_at': '2026-05-26T12:08:37.3056211Z', 'message': {'role': 'assistant', 'content': ''}, 'done': True, 'done_reason': 'stop', 'total_duration': 2081217800, 'load_duration': 26298500, 'prompt_eval_count': 27, 'prompt_eval_duration': 1148560600, 'eval_count': 31, 'eval_duration': 743569100}, id='lc_run--019e642f-ce92-71a0-b6d2-f5bbb84abf6a-0', tool_calls=[], invalid_tool_calls=[])

### Using Pydantic Models

In [16]:
from pydantic import BaseModel, Field

class llm_schema(BaseModel):
    setup: str = Field(description="The setup for the joke")
    punchline: str = Field(description="The punchline for the joke")

In [ ]:
prompt = """
Return ONLY valid JSON like this:
{
  "setup": "string",
  "punchline": "string"
}

Tell me a joke.
"""

response = llm.invoke(prompt)

print("Raw response:", response)

# 3. Parse with Pydantic
try:
    data = json.loads(response.content)
    joke = llm_schema(**data)

    print("\nSetup:", joke.setup)
    print("Punchline:", joke.punchline)

except Exception as e:
    print("Parsing failed:", e)

Raw response: content=' {\n  "ss": "a pun",\n  "punchline": "Why don\'t scientists trust atoms? Because they make up everything!"\n}' additional_kwargs={} response_metadata={'model': 'mistral:latest', 'created_at': '2026-05-26T12:09:28.9945824Z', 'message': {'role': 'assistant', 'content': ''}, 'done': True, 'done_reason': 'stop', 'total_duration': 1244131200, 'load_duration': 32341100, 'prompt_eval_count': 43, 'prompt_eval_duration': 260037300, 'eval_count': 35, 'eval_duration': 746755700} id='lc_run--019e6430-9bc6-76e3-b76f-2374d5eb8a74-0' tool_calls=[] invalid_tool_calls=[]
Parsing failed: 1 validation error for llm_schema
setup
  Field required [type=missing, input_value={'ss': 'a pun', 'punchlin...ey make up everything!"}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


### Using TypedDict

In [6]:
from typing import TypedDict 

class llm_schema_td(TypedDict):
    setup: str
    punchline: str

In [ ]:
prompt = """
Return ONLY valid JSON:
{
  "setup": "string",
  "punchline": "string"
}

Tell me a joke.
"""

response = llm.invoke(prompt)

print("Raw response:", response)


data = json.loads(response)

joke: llm_schema_td = {
    "setup": data["setup"],
    "punchline": data["punchline"]
}

print("\nSetup:", joke["setup"])
print("Punchline:", joke["punchline"])

Raw response:  {
  "setup": "Why don't scientists trust atoms?",
  "punchline": "Because they make up everything!"
}

Setup: Why don't scientists trust atoms?
Punchline: Because they make up everything!


: 